In [2]:
# ============================================================
# CELL 1 — Install packages + environment setup
# ============================================================

!pip install ultralytics --upgrade -q
!pip install realesrgan basicsr -q
!pip install pycocotools -q

# ============================================================
# Patch basicsr torchvision compatibility
# ============================================================

import pathlib
import site

for sp in site.getsitepackages():

    deg = pathlib.Path(sp) / "basicsr/data/degradations.py"

    if deg.exists():

        txt = deg.read_text()

        old = (
            "from torchvision.transforms"
            ".functional_tensor import rgb_to_grayscale"
        )

        new = (
            "from torchvision.transforms"
            ".functional import rgb_to_grayscale"
        )

        if old in txt:

            deg.write_text(
                txt.replace(old, new)
            )

            print("✅ basicsr patched")

        else:

            print("✅ basicsr already patched")

        break

# ============================================================
# Imports
# ============================================================

import os
import gc
import cv2
import torch
import shutil
import numpy as np

# ============================================================
# CUDA setup
# ============================================================

torch.backends.cudnn.benchmark = True

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# ============================================================
# System info
# ============================================================

print(f"\n✅ PyTorch : {torch.__version__}")

if torch.cuda.is_available():

    print(
        f"✅ GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"✅ VRAM    : "
        f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"
    )

# ============================================================
# Disk info
# ============================================================

_, used, free = shutil.disk_usage("/kaggle/working")

print(
    f"✅ Disk    : "
    f"{used/1e9:.1f}GB used | "
    f"{free/1e9:.1f}GB free"
)

# ============================================================
# Test ESRGAN imports
# ============================================================

from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.utils.download_util import load_file_from_url

print("\n✅ ESRGAN imports working")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires nu

In [3]:
# ============================================================
# CELL 2 — All paths, config, imports
# MODIFIED: correct AI-TOD path + verified VisDrone path
# ============================================================

import os, cv2, sys, json, shutil, random, yaml, math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# ── Fix PyTorch 2.6 ──────────────────────────────────────────
if not hasattr(torch, "_load_patched"):
    _orig_load = torch.load
    def patched_load(f, *args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_load(f, *args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────
DEVICE     = torch.device("cuda" if torch.cuda.is_available()
                          else "cpu")
device_obj = DEVICE

# ── VisDrone tiny source ──────────────────────────────────────
VD_ROOT      = Path("/kaggle/input/datasets/arnavadp/"
                    "visdrone-tiny/filtered_tiny")
VD_TRAIN_IMG = VD_ROOT / "images"
VD_TRAIN_LBL = VD_ROOT / "labels"
VD_VAL_IMG   = VD_ROOT / "val_images"
VD_VAL_LBL   = VD_ROOT / "val_labels"

# ── AI-TOD source ─────────────────────────────────────────────
# FIXED: user confirmed path ends at /train
AITOD_TRAIN  = Path("/kaggle/input/datasets/arnavadp/"
                    "aitod-sampled/aitod_sampled/train")

# ── Working directories ───────────────────────────────────────
WORK_DIR     = Path("/kaggle/working")
WEIGHTS_DIR  = WORK_DIR / "esrgan_weights"
VD_TILES     = WORK_DIR / "vd_tiles"
AT_TILES     = WORK_DIR / "aitod_tiles"
RUNS_DIR     = WORK_DIR / "runs"

for d in [WEIGHTS_DIR, VD_TILES, AT_TILES, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Class definitions ─────────────────────────────────────────
VD_CLASSES = ["pedestrian","people","bicycle",
              "car","tricycle","motor"]
VD_NC      = len(VD_CLASSES)

AT_CLASSES = ["airplane","bridge","storage-tank","ship",
              "swimming-pool","vehicle","person","wind-mill"]
AT_NC      = len(AT_CLASSES)

# ── VisDrone raw class ID → YOLO index ───────────────────────
# Used during SAHI slicing (Cell 5)
VD_ID_TO_YOLO = {1:0, 2:1, 3:2, 4:3, 7:4, 10:5}
VD_KEEP_IDS   = set(VD_ID_TO_YOLO.keys())

# ── AI-TOD: detect annotation format during Cell 4 ───────────
# (YOLO txt or COCO json — set after Cell 4 runs)
AT_FORMAT = None   # filled by Cell 4

# ── Hyperparameters ───────────────────────────────────────────
IMG_SIZE    = 640
BATCH_SIZE  = 1
GRAD_ACCUM  = 4
NUM_WORKERS = 2
EPOCHS_2X   = 10
EPOCHS_4X   = 3
LR_ESRGAN   = 1e-4
LR_DETECT   = 1e-4
LAMBDA_SR   = 0.1
NWD_C       = 12.8
MAX_BOXES   = 50
NUM_QUERIES = 100
TILE_H = TILE_W = 640
OVERLAP  = 0.2
MIN_AREA = 0.5

print("✅ Cell 2 config ready")
print(f"   VisDrone  : {VD_ROOT}")
print(f"   AI-TOD    : {AITOD_TRAIN}")
print(f"   Device    : {DEVICE}")

# Verify both source paths exist
for name, path in [("VisDrone", VD_ROOT),
                   ("AI-TOD",   AITOD_TRAIN)]:
    exists = path.exists()
    n = len(list(path.rglob("*"))) if exists else 0
    print(f"   {'✅' if exists else '❌'} {name}: "
          f"{'exists' if exists else 'NOT FOUND'} "
          f"({n} items)")

✅ Cell 2 config ready
   VisDrone  : /kaggle/input/datasets/arnavadp/visdrone-tiny/filtered_tiny
   AI-TOD    : /kaggle/input/datasets/arnavadp/aitod-sampled/aitod_sampled/train
   Device    : cuda
   ✅ VisDrone: exists (1440 items)
   ✅ AI-TOD: exists (3994 items)


In [4]:
# ============================================================
# CELL 3 — Verify VisDrone tiny dataset (FIXED)
# ============================================================

import cv2
from pathlib import Path
from collections import defaultdict

VD_ROOT = Path(
    "/kaggle/input/datasets/arnavadp/visdrone-tiny/filtered_tiny"
)

print("=" * 55)
print("  VisDrone Tiny Dataset Check")
print("=" * 55)

checks = {
    "train images": VD_ROOT / "images",
    "train labels": VD_ROOT / "labels",
    "val images": VD_ROOT / "val_images",
    "val labels": VD_ROOT / "val_labels",
}

# ------------------------------------------------------------
# Folder checks
# ------------------------------------------------------------

for name, d in checks.items():

    if d.exists():

        n = (
            len(list(d.glob("*.jpg"))) +
            len(list(d.glob("*.png"))) +
            len(list(d.glob("*.txt")))
        )

    else:
        n = 0

    status = "✅" if n > 0 else "❌"

    print(f"  {status}  {name:<15}: {n} files  [{d.name}]")

# ------------------------------------------------------------
# Sample label check
# ------------------------------------------------------------

sample_lbl = next((VD_ROOT / "labels").glob("*.txt"), None)

if sample_lbl:

    lines = sample_lbl.read_text().strip().split("\n")

    print(f"\n  Sample label ({sample_lbl.name}):")

    for l in lines[:3]:
        print(f"    '{l}'")

    cls_ids = set()

    for l in lines:

        # VisDrone format:
        # x,y,w,h,score,class,truncation,occlusion

        p = l.strip().split(",")

        if len(p) >= 6:

            cls_id = int(p[5])
            cls_ids.add(cls_id)

    print(f"\n  Class IDs in this file: {sorted(cls_ids)}")

# ------------------------------------------------------------
# Sample image size
# ------------------------------------------------------------

sample_img = next((VD_ROOT / "images").glob("*.jpg"), None)

if sample_img:

    img = cv2.imread(str(sample_img))

    print(
        f"\n  Sample image size: "
        f"{img.shape[1]} × {img.shape[0]}"
    )

# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------

print(f"\n  Class distribution (train):")

VISDRONE_CLASSES = {
    1: "pedestrian",
    2: "people",
    3: "bicycle",
    4: "car",
    5: "van",
    6: "truck",
    7: "tricycle",
    8: "awning-tricycle",
    9: "bus",
    10: "motor"
}

counts = defaultdict(int)

for lbl in (VD_ROOT / "labels").glob("*.txt"):

    lines = lbl.read_text().strip().split("\n")

    for line in lines:

        p = line.strip().split(",")

        if len(p) >= 6:

            cls_id = int(p[5])

            counts[cls_id] += 1

# Print distribution
for cls_id, cls_name in VISDRONE_CLASSES.items():

    print(
        f"    {cls_id:>2} "
        f"{cls_name:<18}: "
        f"{counts[cls_id]:>6,}"
    )

print("=" * 55)

  VisDrone Tiny Dataset Check
  ✅  train images   : 658 files  [images]
  ✅  train labels   : 658 files  [labels]
  ✅  val images     : 60 files  [val_images]
  ✅  val labels     : 60 files  [val_labels]

  Sample label (0000348_02157_d_0000417.txt):
    '234,358,17,37,1,1,0,0'
    '248,377,11,22,1,1,0,0'
    '650,149,11,20,1,1,0,0'

  Class IDs in this file: [1, 2]

  Sample image size: 1400 × 1050

  Class distribution (train):
     1 pedestrian        : 13,027
     2 people            :  2,519
     3 bicycle           :    841
     4 car               :  4,236
     5 van               :      0
     6 truck             :      0
     7 tricycle          :    161
     8 awning-tricycle   :      0
     9 bus               :      0
    10 motor             :  1,435


In [5]:
# ============================================================
# CELL 4 — Inspect AI-TOD dataset structure
# MODIFIED: uses correct AITOD_TRAIN path
#           detects annotation format automatically
# ============================================================

from pathlib import Path
import json

AITOD_TRAIN = Path("/kaggle/input/datasets/arnavadp/"
                   "aitod-sampled/aitod_sampled/train")

print("="*65)
print(f"AI-TOD TRAIN: {AITOD_TRAIN}")
print("="*65)

if not AITOD_TRAIN.exists():
    print("❌ Path does not exist!")
    print("   Check dataset is attached to this notebook")
else:
    # Show all items (depth limited)
    all_items = sorted(AITOD_TRAIN.rglob("*"))
    print(f"Total items: {len(all_items)}\n")
    for p in all_items[:40]:
        typ = "DIR " if p.is_dir() else "FILE"
        print(f"  {typ} {p.relative_to(AITOD_TRAIN)}")
    if len(all_items) > 40:
        print(f"  ... and {len(all_items)-40} more")

    # ── Detect format ─────────────────────────────────────────
    print("\n" + "="*65)
    print("FORMAT DETECTION")
    print("="*65)

    json_files = list(AITOD_TRAIN.rglob("*.json"))
    txt_files  = list(AITOD_TRAIN.rglob("*.txt"))
    img_files  = (list(AITOD_TRAIN.rglob("*.jpg")) +
                  list(AITOD_TRAIN.rglob("*.png")))

    print(f"  JSON files : {len(json_files)}")
    print(f"  TXT files  : {len(txt_files)}")
    print(f"  Images     : {len(img_files)}")

    # COCO JSON format
    if json_files:
        print("\n  → COCO JSON format detected")
        with open(json_files[0]) as f:
            coco = json.load(f)
        print(f"  Images      : {len(coco.get('images',[]))}")
        print(f"  Annotations : {len(coco.get('annotations',[]))}")
        cats = coco.get("categories",[])
        print(f"  Categories  : {[(c['id'],c['name']) for c in cats]}")
        print(f"\n  JSON path   : {json_files[0]}")
        AT_FORMAT = "coco"

    # YOLO TXT format
    elif txt_files:
        sample = txt_files[0]
        sample_line = sample.read_text().strip().split("\n")[0]
        print(f"\n  → YOLO TXT format detected")
        print(f"  Sample line : '{sample_line}'")
        parts = sample_line.strip().split()
        if len(parts) == 5:
            print("  ✅ Confirmed: class cx cy w h format")
        AT_FORMAT = "yolo"
    else:
        print("\n  ❌ No annotation files found")
        AT_FORMAT = None

    # Image folder
    if img_files:
        import cv2
        sample_img = cv2.imread(str(img_files[0]))
        if sample_img is not None:
            print(f"\n  Sample image: {img_files[0].name}")
            print(f"  Image size  : {sample_img.shape[1]}"
                  f"×{sample_img.shape[0]}")

print(f"\n✅ AT_FORMAT = '{AT_FORMAT}'")
print("   (used by Cell 5 SAHI slicer)")

AI-TOD TRAIN: /kaggle/input/datasets/arnavadp/aitod-sampled/aitod_sampled/train
Total items: 3994

  DIR  images
  FILE images/0000042_02231_d_0000075__160_0_png.rf.50468a58af59a28c100bef261c5fda8c.jpg
  FILE images/0000043_00500_d_0000077__0_0_png.rf.ea494cd873f88155ade767fcb68970aa.jpg
  FILE images/0000101_01577_d_0000018__160_0_png.rf.7d90521e557453a3e2485fcfd48354bd.jpg
  FILE images/0000130_01308_d_0000139__1200_0_png.rf.514f5a65108322e65ab9bcd7b0051986.jpg
  FILE images/0000142_04458_d_0000045__1120_0_png.rf.c70191bce6d7b3e9df1a75bf749337a4.jpg
  FILE images/0000143_00681_d_0000051__1120_0_png.rf.148495fb41e4df63bebeda0668bc9307.jpg
  FILE images/0000154_00001_d_0000001__160_0_png.rf.551fe65441efffe8549758ecd160563c.jpg
  FILE images/0000170_00001_d_0000001__0_0_png.rf.e8ed3fea099dac5f7db6c1614f2772e1.jpg
  FILE images/0000170_00801_d_0000001__160_0_png.rf.362d4343a1e0c7664a56e2ca5dca507a.jpg
  FILE images/0000170_01201_d_0000001__160_0_png.rf.684e498bce3a92fb9f6b362d1ee9a76a.jp

In [6]:
# ============================================================
# CELL 5 — SAHI Slicing for VisDrone + AI-TOD
# MODIFIED:
#   ✅ VD uses raw VisDrone CSV format (x,y,w,h,score,cls...)
#   ✅ AT uses detected format (COCO JSON or YOLO TXT)
#   ✅ Both output clean YOLO-format tiles
#   ✅ Only tiles WITH objects are saved (no empty tiles)
# ============================================================

import cv2, shutil, json
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

VD_ROOT     = Path("/kaggle/input/datasets/arnavadp/"
                   "visdrone-tiny/filtered_tiny")
AITOD_TRAIN = Path("/kaggle/input/datasets/arnavadp/"
                   "aitod-sampled/aitod_sampled/train")
VD_TILES    = Path("/kaggle/working/vd_tiles")
AT_TILES    = Path("/kaggle/working/aitod_tiles")

TILE_H   = TILE_W = 640
OVERLAP  = 0.2
MIN_AREA = 0.5

# VisDrone: keep only these raw class IDs
VD_ID_TO_YOLO = {1:0, 2:1, 3:2, 4:3, 7:4, 10:5}
VD_KEEP_IDS   = set(VD_ID_TO_YOLO.keys())

# AI-TOD: COCO category_id (1-indexed) → YOLO index
AT_CAT_TO_YOLO = {1:0,2:1,3:2,4:3,5:4,6:5,7:6,8:7}


# ============================================================
# Label parsers → return list of [yolo_cls, x1, y1, x2, y2]
#                 in PIXEL coordinates
# ============================================================

def parse_visdrone_label(lbl_path, W, H):
    """Parse VisDrone CSV annotation → pixel xyxy boxes."""
    boxes = []
    if not lbl_path.exists():
        return boxes
    for line in lbl_path.read_text().strip().split("\n"):
        p = line.strip().split(",")
        if len(p) < 6:
            continue
        score  = int(p[4])
        cls_id = int(p[5])
        if score == 0:           # ignored region
            continue
        if cls_id not in VD_KEEP_IDS:
            continue
        x, y, w, h = float(p[0]),float(p[1]),float(p[2]),float(p[3])
        if w <= 1 or h <= 1:
            continue
        yolo_cls = VD_ID_TO_YOLO[cls_id]
        boxes.append([yolo_cls, x, y, x+w, y+h])
    return boxes


def parse_yolo_label(lbl_path, W, H):
    """Parse YOLO txt annotation → pixel xyxy boxes."""
    boxes = []
    if not lbl_path.exists():
        return boxes
    for line in lbl_path.read_text().strip().split("\n"):
        p = line.strip().split()
        if len(p) != 5:
            continue
        cls = int(p[0])
        cx  = float(p[1]) * W;  cy = float(p[2]) * H
        bw  = float(p[3]) * W;  bh = float(p[4]) * H
        boxes.append([cls, cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2])
    return boxes


def build_coco_lookup(json_path):
    """Build image_id → list of [yolo_cls,x1,y1,x2,y2] dict."""
    with open(json_path) as f:
        coco = json.load(f)
    # cat_id → yolo_cls
    cats = {c["id"]: AT_CAT_TO_YOLO.get(c["id"], c["id"]-1)
            for c in coco.get("categories",[])}
    lookup = defaultdict(list)
    for ann in coco.get("annotations", []):
        img_id = ann["image_id"]
        cat_id = ann["category_id"]
        x, y, bw, bh = ann["bbox"]
        if bw <= 1 or bh <= 1:
            continue
        yolo_cls = cats.get(cat_id, 0)
        lookup[img_id].append([yolo_cls, x, y, x+bw, y+bh])
    # image_id → filename
    id_to_file = {img["id"]: img["file_name"]
                  for img in coco.get("images",[])}
    return lookup, id_to_file


# ============================================================
# Tile slicer
# ============================================================

def slice_image_to_tiles(img, boxes_xyxy,
                          out_img_dir, out_lbl_dir,
                          stem, tile_h, tile_w,
                          overlap, min_area):
    """Slice one image into tiles. Save only tiles with objects."""
    H, W = img.shape[:2]
    step_h = int(tile_h * (1 - overlap))
    step_w = int(tile_w * (1 - overlap))
    saved  = 0

    for y in range(0, max(1, H-tile_h+step_h), step_h):
        for x in range(0, max(1, W-tile_w+step_w), step_w):
            y2 = min(y+tile_h, H); x2 = min(x+tile_w, W)
            y1 = max(0, y2-tile_h); x1 = max(0, x2-tile_w)
            tile = img[y1:y2, x1:x2]
            th, tw = tile.shape[:2]

            tile_boxes = []
            for (cls, bx1, by1, bx2, by2) in boxes_xyxy:
                # Intersect
                ix1 = max(bx1, x1); iy1 = max(by1, y1)
                ix2 = min(bx2, x2); iy2 = min(by2, y2)
                if ix2 <= ix1 or iy2 <= iy1:
                    continue
                orig  = max(1, (bx2-bx1)*(by2-by1))
                inter = (ix2-ix1)*(iy2-iy1)
                if inter/orig < min_area:
                    continue
                # To normalised YOLO
                ncx = ((ix1+ix2)/2 - x1) / tw
                ncy = ((iy1+iy2)/2 - y1) / th
                nw  = (ix2-ix1) / tw
                nh  = (iy2-iy1) / th
                ncx = max(0.001, min(0.999, ncx))
                ncy = max(0.001, min(0.999, ncy))
                nw  = max(0.001, min(0.999, nw))
                nh  = max(0.001, min(0.999, nh))
                tile_boxes.append(
                    f"{int(cls)} {ncx:.6f} {ncy:.6f}"
                    f" {nw:.6f} {nh:.6f}")

            if not tile_boxes:
                continue    # skip empty tiles

            tstem = f"{stem}_{y1}_{x1}"
            cv2.imwrite(str(out_img_dir/f"{tstem}.jpg"), tile)
            (out_lbl_dir/f"{tstem}.txt").write_text(
                "\n".join(tile_boxes))
            saved += 1

    return saved


# ============================================================
# Run VisDrone slicing (train + val)
# ============================================================

print("="*60)
print("SLICING VISDRONE TINY")
print("="*60)

for split, img_src, lbl_src in [
    ("train", VD_ROOT/"images",     VD_ROOT/"labels"),
    ("val",   VD_ROOT/"val_images", VD_ROOT/"val_labels"),
]:
    dst_img = VD_TILES / split / "images"
    dst_lbl = VD_TILES / split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    imgs   = sorted(img_src.glob("*.jpg"))
    total_tiles = obj_tiles = 0

    print(f"\n[VisDrone {split}] {len(imgs)} images")

    for img_path in tqdm(imgs, desc=f"VD {split}"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]
        lbl_path = lbl_src / (img_path.stem + ".txt")
        boxes = parse_visdrone_label(lbl_path, W, H)
        if not boxes:
            continue
        n = slice_image_to_tiles(
            img, boxes, dst_img, dst_lbl,
            img_path.stem, TILE_H, TILE_W,
            OVERLAP, MIN_AREA)
        obj_tiles += n

    n_img = len(list(dst_img.glob("*.jpg")))
    print(f"  ✅ {obj_tiles} object tiles saved")


# ============================================================
# Run AI-TOD slicing (train only — no val split in source)
# ============================================================

print("\n" + "="*60)
print("SLICING AI-TOD")
print("="*60)

# Detect format
json_files = list(AITOD_TRAIN.rglob("*.json"))
txt_files  = list(AITOD_TRAIN.rglob("*.txt"))

# Output: train split (use 90/10 split for val)
AT_TRAIN_IMG = AT_TILES / "train" / "images"
AT_TRAIN_LBL = AT_TILES / "train" / "labels"
AT_VAL_IMG   = AT_TILES / "val"   / "images"
AT_VAL_LBL   = AT_TILES / "val"   / "labels"
for d in [AT_TRAIN_IMG, AT_TRAIN_LBL, AT_VAL_IMG, AT_VAL_LBL]:
    d.mkdir(parents=True, exist_ok=True)

import random as rng
rng.seed(42)
all_tiles_stemmed = []    # collect stems for train/val split

if json_files:
    # COCO JSON format
    print(f"  Format: COCO JSON ({json_files[0].name})")
    coco_lookup, id_to_file = build_coco_lookup(json_files[0])

    # Find image directory
    img_dirs = [d for d in AITOD_TRAIN.rglob("*")
                if d.is_dir() and any(d.glob("*.jpg"))]
    img_dir  = img_dirs[0] if img_dirs else AITOD_TRAIN

    print(f"  Image dir  : {img_dir}")
    print(f"  Images with anns: {len(coco_lookup)}")

    # Use a temp dir to collect all tiles before split
    TMP_IMG = AT_TILES / "_tmp" / "images"
    TMP_LBL = AT_TILES / "_tmp" / "labels"
    TMP_IMG.mkdir(parents=True, exist_ok=True)
    TMP_LBL.mkdir(parents=True, exist_ok=True)

    for img_id, boxes in tqdm(coco_lookup.items(),
                               desc="COCO tiles"):
        fname    = id_to_file.get(img_id, "")
        img_path = img_dir / Path(fname).name
        if not img_path.exists():
            candidates = list(img_dir.rglob(Path(fname).name))
            if not candidates:
                continue
            img_path = candidates[0]
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        n = slice_image_to_tiles(
            img, boxes, TMP_IMG, TMP_LBL,
            img_path.stem, TILE_H, TILE_W,
            OVERLAP, MIN_AREA)
        all_tiles_stemmed.extend(
            [p.stem for p in TMP_IMG.glob(f"{img_path.stem}_*.jpg")])

elif txt_files:
    # YOLO TXT format
    img_dirs = [d for d in AITOD_TRAIN.rglob("*")
                if d.is_dir() and any(d.glob("*.jpg"))]
    lbl_dirs = [d for d in AITOD_TRAIN.rglob("*")
                if d.is_dir() and any(d.glob("*.txt"))]
    img_dir  = img_dirs[0] if img_dirs else AITOD_TRAIN
    lbl_dir  = lbl_dirs[0] if lbl_dirs else AITOD_TRAIN

    print(f"  Format: YOLO TXT")
    print(f"  Image dir: {img_dir}")

    TMP_IMG = AT_TILES / "_tmp" / "images"
    TMP_LBL = AT_TILES / "_tmp" / "labels"
    TMP_IMG.mkdir(parents=True, exist_ok=True)
    TMP_LBL.mkdir(parents=True, exist_ok=True)

    for img_path in tqdm(sorted(img_dir.glob("*.jpg")),
                          desc="YOLO tiles"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        boxes = parse_yolo_label(lbl_path, W, H)
        if not boxes:
            continue
        slice_image_to_tiles(
            img, boxes, TMP_IMG, TMP_LBL,
            img_path.stem, TILE_H, TILE_W,
            OVERLAP, MIN_AREA)
else:
    print("  ❌ No annotations found in AI-TOD path")
    TMP_IMG = None

# ── 90/10 train/val split of all AI-TOD tiles ────────────────
if TMP_IMG and TMP_IMG.exists():
    all_tile_imgs = sorted(TMP_IMG.glob("*.jpg"))
    rng.shuffle(all_tile_imgs)
    n_val = max(1, int(len(all_tile_imgs) * 0.1))
    val_stems = {p.stem for p in all_tile_imgs[:n_val]}

    for tile_img in all_tile_imgs:
        tile_lbl = TMP_LBL / (tile_img.stem + ".txt")
        split    = "val" if tile_img.stem in val_stems else "train"
        dst_i    = AT_TILES / split / "images" / tile_img.name
        dst_l    = AT_TILES / split / "labels" / (tile_img.stem+".txt")
        if not dst_i.exists():
            shutil.copy(tile_img, dst_i)
        if tile_lbl.exists() and not dst_l.exists():
            shutil.copy(tile_lbl, dst_l)

    # Cleanup tmp
    shutil.rmtree(AT_TILES / "_tmp", ignore_errors=True)

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*60)
print("SAHI SLICING COMPLETE")
print("="*60)
for name, base in [("VisDrone", VD_TILES), ("AI-TOD", AT_TILES)]:
    for split in ["train","val"]:
        d = base / split / "images"
        n = len(list(d.glob("*.jpg"))) if d.exists() else 0
        print(f"  {name:<10} [{split}]: {n:>6} tiles")

import shutil as sh
_, used, free = sh.disk_usage("/kaggle/working")
print(f"\n  Disk: {used/1e9:.1f}GB used  {free/1e9:.1f}GB free")

SLICING VISDRONE TINY

[VisDrone train] 658 images


VD train:   0%|          | 0/658 [00:00<?, ?it/s]

  ✅ 3459 object tiles saved

[VisDrone val] 60 images


VD val:   0%|          | 0/60 [00:00<?, ?it/s]

  ✅ 259 object tiles saved

SLICING AI-TOD
  Format: YOLO TXT
  Image dir: /kaggle/input/datasets/arnavadp/aitod-sampled/aitod_sampled/train/images


YOLO tiles:   0%|          | 0/1996 [00:00<?, ?it/s]


SAHI SLICING COMPLETE
  VisDrone   [train]:   3459 tiles
  VisDrone   [val]:    259 tiles
  AI-TOD     [train]:   5830 tiles
  AI-TOD     [val]:    647 tiles

  Disk: 1.2GB used  19.7GB free


In [7]:
# ============================================================
# CELL 6 — Download ESRGAN weights + define NWD loss classes
# ============================================================

import urllib.request, torch, torch.nn as nn
from pathlib import Path
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

WEIGHTS_DIR = Path("/kaggle/working/esrgan_weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

ESRGAN_WEIGHTS = {
    "2x": {
        "url"  : "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
        "path" : WEIGHTS_DIR / "RealESRGAN_x2plus.pth",
        "scale": 2,
    },
    "4x": {
        "url"  : "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        "path" : WEIGHTS_DIR / "RealESRGAN_x4plus.pth",
        "scale": 4,
    },
}

for key, cfg in ESRGAN_WEIGHTS.items():
    if not cfg["path"].exists():
        print(f"⬇️  Downloading ESRGAN {key}...")
        urllib.request.urlretrieve(cfg["url"], cfg["path"])
        mb = cfg["path"].stat().st_size / 1e6
        print(f"   ✅ {mb:.0f} MB")
    else:
        mb = cfg["path"].stat().st_size / 1e6
        print(f"✅ ESRGAN {key} ready ({mb:.0f} MB)")


# ── NWD Loss ─────────────────────────────────────────────────
class NWDLoss(nn.Module):
    """
    Normalized Wasserstein Distance loss.
    Designed specifically for tiny object detection.

    Models each box as a 2D Gaussian:
      μ  = (cx, cy)    — centre
      σ  = (w/2, h/2)  — spread

    W2² = ||μ1-μ2||² + ||σ1-σ2||²
    NWD = exp(-sqrt(W2²) / C)
    Loss= 1 - NWD

    Advantage over IoU: smooth gradient even when
    boxes do not overlap (common for tiny objects).
    """
    def __init__(self, C: float = 12.8):
        super().__init__()
        self.C = C

    def forward(self, pred, target,
                reduction="mean"):
        mu1 = pred[:,:2];   sigma1 = pred[:,2:]/2
        mu2 = target[:,:2]; sigma2 = target[:,2:]/2
        center_d = ((mu1-mu2)**2).sum(-1)
        sigma_d  = ((sigma1-sigma2)**2).sum(-1)
        w2       = torch.sqrt(center_d+sigma_d+1e-7)
        nwd      = torch.exp(-w2/self.C)
        loss     = 1.0 - nwd
        if reduction=="mean":  return loss.mean()
        if reduction=="sum":   return loss.sum()
        return loss


class SRQualityLoss(nn.Module):
    """L1 pixel loss + VGG19 perceptual loss for SR quality."""
    def __init__(self, device):
        super().__init__()
        import torchvision.models as tv
        vgg = tv.vgg19(weights=tv.VGG19_Weights.IMAGENET1K_V1)
        self.feat = nn.Sequential(
            *list(vgg.features)[:18]).eval().to(device)
        for p in self.feat.parameters():
            p.requires_grad = False
        self.l1   = nn.L1Loss()
        self.mean = torch.tensor(
            [0.485,0.456,0.406]).view(1,3,1,1).to(device)
        self.std  = torch.tensor(
            [0.229,0.224,0.225]).view(1,3,1,1).to(device)

    def forward(self, sr, hr):
        L_pix  = self.l1(sr, hr)
        sr_n   = (sr-self.mean)/self.std
        hr_n   = (hr-self.mean)/self.std
        L_perc = self.l1(self.feat(sr_n), self.feat(hr_n))
        return L_pix + 0.1*L_perc


def build_esrgan(scale, device):
    """Build trainable ESRGAN model."""
    model = RRDBNet(num_in_ch=3,num_out_ch=3,
                    num_feat=64,num_block=23,
                    num_grow_ch=32,scale=scale)
    upsampler = RealESRGANer(
        scale      = scale,
        model_path = str(ESRGAN_WEIGHTS[f"{scale}x"]["path"]),
        model      = model,
        tile       = 0,
        tile_pad   = 0,
        pre_pad    = 0,
        half       = False,
        device     = device,
    )
    upsampler.model.train()
    for p in upsampler.model.parameters():
        p.requires_grad = True
    return upsampler


device_obj = torch.device(f"cuda:0"
                           if torch.cuda.is_available() else "cpu")
print("\n✅ NWD Loss, SRQualityLoss, build_esrgan ready")

⬇️  Downloading ESRGAN 2x...
   ✅ 67 MB
⬇️  Downloading ESRGAN 4x...
   ✅ 67 MB

✅ NWD Loss, SRQualityLoss, build_esrgan ready


In [8]:
# ============================================================
# CELL 7 — JointTileDataset
# MODIFIED:
#   ✅ Removed wrong VisDrone remapping
#      (tiles from Cell 5 are already in YOLO format)
#   ✅ Reads YOLO format labels directly
#   ✅ Returns properly padded box tensor
# ============================================================

import cv2, torch, random
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset


class JointTileDataset(Dataset):
    """
    Loads 640×640 YOLO-format tiles produced by Cell 5.

    Returns per item:
      lr  : (3, LR_SIZE, LR_SIZE) — ESRGAN input
      hr  : (3, 640, 640)         — SR quality reference
      boxes: (MAX_BOXES, 5)       — [cls, cx, cy, w, h]
              cls=-1 means padding row
    """

    def __init__(self,
                 img_dir: Path,
                 lbl_dir: Path,
                 sr_scale: int = 2,
                 max_boxes: int = 50):

        self.img_dir   = Path(img_dir)
        self.lbl_dir   = Path(lbl_dir)
        self.hr_size   = 640
        self.lr_size   = 640 // sr_scale   # 320 or 160
        self.max_boxes = max_boxes

        # Only include tiles that have at least one object
        all_imgs = sorted(self.img_dir.glob("*.jpg"))
        self.imgs = []
        for p in all_imgs:
            lbl = self.lbl_dir / (p.stem + ".txt")
            if lbl.exists() and lbl.read_text().strip():
                self.imgs.append(p)

        print(f"   [{self.img_dir.parent.name}/"
              f"{self.img_dir.name}]: "
              f"{len(self.imgs)} object tiles  "
              f"(LR {self.lr_size}×{self.lr_size})")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        lbl_path = self.lbl_dir / (img_path.stem + ".txt")

        # ── Load image ────────────────────────────────────────
        img = cv2.imread(str(img_path))
        if img is None:
            img = np.zeros((self.hr_size, self.hr_size, 3),
                           np.uint8)
        img_rgb = cv2.cvtColor(
            cv2.resize(img, (self.hr_size, self.hr_size),
                       interpolation=cv2.INTER_LINEAR),
            cv2.COLOR_BGR2RGB)

        # ── HR tensor ─────────────────────────────────────────
        hr = (torch.tensor(img_rgb, dtype=torch.float32)
              .permute(2, 0, 1) / 255.0)

        # ── LR tensor ─────────────────────────────────────────
        lr_img = cv2.resize(img_rgb,
                            (self.lr_size, self.lr_size),
                            interpolation=cv2.INTER_AREA)
        lr = (torch.tensor(lr_img, dtype=torch.float32)
              .permute(2, 0, 1) / 255.0)

        # ── Labels (already YOLO format from Cell 5) ─────────
        # format per line: cls cx cy w h  (all normalised)
        box_tensor = torch.full((self.max_boxes, 5), -1.0)
        count = 0

        if lbl_path.exists():
            for line in lbl_path.read_text().strip().split("\n"):
                p = line.strip().split()
                if len(p) != 5 or count >= self.max_boxes:
                    continue
                try:
                    cls = int(p[0])
                    cx, cy, bw, bh = (float(p[1]), float(p[2]),
                                      float(p[3]), float(p[4]))
                    # Basic sanity check
                    if not (0 < bw < 1 and 0 < bh < 1):
                        continue
                    box_tensor[count] = torch.tensor(
                        [cls, cx, cy, bw, bh])
                    count += 1
                except ValueError:
                    continue

        return {"lr": lr, "hr": hr, "boxes": box_tensor}

In [9]:
# ── Add this disk space guard to the top of Cell 8 ───────────
import shutil as sh

def check_disk_gb():
    _, _, free = sh.disk_usage("/kaggle/working")
    return free / 1e9

# Add inside the loop in apply_esrgan_to_dataset():
# if check_disk_gb() < 2.0:
#     print(f"⚠️  Only {check_disk_gb():.1f}GB left — stopping!")
#     break

In [10]:
# ============================================================
# DOWNLOAD ESRGAN WEIGHTS
# ============================================================

import os
from pathlib import Path
from basicsr.utils.download_util import load_file_from_url

ROOT = Path("/root/.cache/realesrgan")

ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# URLs
# ------------------------------------------------------------

URLS = {

    "x2": (
        "https://github.com/xinntao/Real-ESRGAN/"
        "releases/download/v0.2.1/"
        "RealESRGAN_x2plus.pth"
    ),

    "x4": (
        "https://github.com/xinntao/Real-ESRGAN/"
        "releases/download/v0.1.0/"
        "RealESRGAN_x4plus.pth"
    ),
}

# ------------------------------------------------------------
# Download
# ------------------------------------------------------------

for name, url in URLS.items():

    print(f"\n⬇️ Downloading {name} weights...")

    path = load_file_from_url(

        url=url,

        model_dir=str(ROOT),

        progress=True,
    )

    print(f"✅ Saved to: {path}")

print("\n🎉 ESRGAN weights ready")


⬇️ Downloading x2 weights...
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth" to /root/.cache/realesrgan/RealESRGAN_x2plus.pth



100%|██████████| 64.0M/64.0M [00:00<00:00, 241MB/s]


✅ Saved to: /root/.cache/realesrgan/RealESRGAN_x2plus.pth

⬇️ Downloading x4 weights...
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" to /root/.cache/realesrgan/RealESRGAN_x4plus.pth



100%|██████████| 63.9M/63.9M [00:00<00:00, 266MB/s]

✅ Saved to: /root/.cache/realesrgan/RealESRGAN_x4plus.pth

🎉 ESRGAN weights ready


In [11]:
# ============================================================
# TEST ESRGAN
# ============================================================

from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

scale = 2

model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=scale
)

upsampler = RealESRGANer(
    scale=scale,
    model_path="/root/.cache/realesrgan/RealESRGAN_x2plus.pth",
    model=model,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=True,
    device="cuda"
)

print("✅ ESRGAN LOADED SUCCESSFULLY")

✅ ESRGAN LOADED SUCCESSFULLY


In [12]:
# ============================================================
# CELL 10 — Evaluate all 4 trained models using ultralytics val
# ============================================================

import yaml, torch
from ultralytics import RTDETR
from pathlib import Path

VD_TILES = Path("/kaggle/working/vd_tiles")
AT_TILES = Path("/kaggle/working/aitod_tiles")
RUNS_DIR = Path("/kaggle/working/runs")
WORK_DIR = Path("/kaggle/working")

VD_CLASSES = ["pedestrian","people","bicycle",
              "car","tricycle","motor"]
AT_CLASSES = ["airplane","bridge","storage-tank","ship",
              "swimming-pool","vehicle","person","wind-mill"]

EVAL_CONFIGS = [
    {"name":"VisDrone_2x","tile_dir":VD_TILES,
     "nc":6,"names":VD_CLASSES},
    {"name":"VisDrone_4x","tile_dir":VD_TILES,
     "nc":6,"names":VD_CLASSES},
    {"name":"AITOD_2x","tile_dir":AT_TILES,
     "nc":8,"names":AT_CLASSES},
    {"name":"AITOD_4x","tile_dir":AT_TILES,
     "nc":8,"names":AT_CLASSES},
]

RESULTS = {}

for cfg in EVAL_CONFIGS:
    name     = cfg["name"]
    ckpt_pt  = RUNS_DIR / name / "best.pt"

    print(f"\n📊 Evaluating {name}...")

    if not ckpt_pt.exists():
        print(f"  ❌ Checkpoint not found: {ckpt_pt}")
        RESULTS[name] = {
            "mAP50":0,"mAP50-95":0,"Precision":0,"Recall":0}
        continue

    # Create YAML
    yaml_path = WORK_DIR / f"{name}_eval.yaml"
    with open(yaml_path,"w") as f:
        yaml.dump({
            "path" : str(cfg["tile_dir"]),
            "train": "train/images",
            "val"  : "val/images",
            "nc"   : cfg["nc"],
            "names": cfg["names"],
        }, f, default_flow_style=False)

    try:
        # Load model with saved weights
        ckpt  = torch.load(str(ckpt_pt),
                           map_location="cpu",
                           weights_only=False)

        # Create RT-DETR and load state
        model = RTDETR("rtdetr-l.pt")
        if "rtdetr" in ckpt:
            try:
                model.model.load_state_dict(
                    ckpt["rtdetr"], strict=False)
                print(f"  ✅ Loaded joint-trained weights")
            except Exception as e:
                print(f"  ⚠️  Weight load: {e}")

        metrics = model.val(
            data    = str(yaml_path),
            split   = "val",
            imgsz   = 640,
            batch   = BATCH_SIZE,
            device  = 0,
            verbose = False,
        )

        RESULTS[name] = {
            "mAP50"    : metrics.box.map50,
            "mAP50-95" : metrics.box.map,
            "Precision": metrics.box.mp,
            "Recall"   : metrics.box.mr,
        }

        print(f"  mAP@50    : {metrics.box.map50:.4f}")
        print(f"  mAP@50-95 : {metrics.box.map:.4f}")
        print(f"  Precision : {metrics.box.mp:.4f}")
        print(f"  Recall    : {metrics.box.mr:.4f}")

    except Exception as e:
        print(f"  ❌ Error: {e}")
        RESULTS[name] = {
            "mAP50":0,"mAP50-95":0,"Precision":0,"Recall":0}

# Print full table
print("\n" + "="*65)
print("  ALL RESULTS — Joint ESRGAN + RT-DETR + NWD Loss")
print("="*65)
print(f"  {'Experiment':<18} {'mAP@50':>8} "
      f"{'mAP@50-95':>10} {'P':>8} {'R':>8}")
print("  " + "-"*55)
for k, v in RESULTS.items():
    print(f"  {k:<18} {v['mAP50']:>8.4f} "
          f"{v['mAP50-95']:>10.4f} "
          f"{v['Precision']:>8.4f} "
          f"{v['Recall']:>8.4f}")
print("="*65)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

📊 Evaluating VisDrone_2x...
  ❌ Checkpoint not found: /kaggle/working/runs/VisDrone_2x/best.pt

📊 Evaluating VisDrone_4x...
  ❌ Checkpoint not found: /kaggle/working/runs/VisDrone_4x/best.pt

📊 Evaluating AITOD_2x...
  ❌ Checkpoint not found: /kaggle/working/runs/AITOD_2x/best.pt

📊 Evaluating AITOD_4x...
  ❌ Checkpoint not found: /kaggle/working/runs/AITOD_4x/best.pt

  ALL RESULTS — Joint ESRGAN + RT-DETR + NWD Loss
  Experiment           mAP@50  mAP@50-95        P        R
  -------------------------------------------------------
  VisDrone_2x          0.0000     0.0000   0.0000   0.0000
  VisDrone_4x          0.0000     0.0000   0.0000   0.0000
  AITOD_2x             0.0000  

In [13]:
# ============================================================
# CELL 11 — Training setup: NWD + DetectionHead + Trainer
# MODIFIED:
#   ✅ Correct AT_TILES source
#   ✅ VisDrone runs FIRST then AI-TOD (as requested)
#   ✅ Proper ESRGAN loading (half=False)
#   ✅ Real NWD loss
#   ✅ Real detection head
# ============================================================

import os, gc, cv2, torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

torch.backends.cudnn.benchmark = True
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VD_TILES    = Path("/kaggle/working/vd_tiles")
AT_TILES    = Path("/kaggle/working/aitod_tiles")
RUNS_DIR    = Path("/kaggle/working/runs")
RUNS_DIR.mkdir(exist_ok=True)

IMG_SIZE    = 640
BATCH_SIZE  = 1
GRAD_ACCUM  = 4
NUM_WORKERS = 2
EPOCHS_2X   = 10
EPOCHS_4X   = 3
LR_ESRGAN   = 1e-4
LR_DETECT   = 1e-4
LAMBDA_SR   = 0.1
NWD_C       = 12.8
MAX_BOXES   = 50
NUM_QUERIES = 100

# ── ORDER: VisDrone first, then AI-TOD ───────────────────────
EXPERIMENTS = [
    {"name":"VisDrone_2x","tile_dir":VD_TILES,
     "sr_scale":2,"epochs":EPOCHS_2X,"num_classes":6},
    {"name":"VisDrone_4x","tile_dir":VD_TILES,
     "sr_scale":4,"epochs":EPOCHS_4X,"num_classes":6},
    {"name":"AITOD_2x",   "tile_dir":AT_TILES,
     "sr_scale":2,"epochs":EPOCHS_2X,"num_classes":8},
    {"name":"AITOD_4x",   "tile_dir":AT_TILES,
     "sr_scale":4,"epochs":EPOCHS_4X,"num_classes":8},
]

# ── True NWD loss ─────────────────────────────────────────────
def nwd_loss(pred_boxes, gt_boxes, C=12.8):
    if pred_boxes.shape[0] == 0 or gt_boxes.shape[0] == 0:
        return torch.tensor(0.0, device=pred_boxes.device,
                            requires_grad=True)
    mu_p  = pred_boxes[:, :2];   sig_p = pred_boxes[:, 2:] / 2
    mu_g  = gt_boxes[:, :2];     sig_g = gt_boxes[:, 2:] / 2
    cd = ((mu_p.unsqueeze(1) - mu_g.unsqueeze(0))**2).sum(-1)
    sd = ((sig_p.unsqueeze(1) - sig_g.unsqueeze(0))**2).sum(-1)
    w2 = torch.sqrt(cd + sd + 1e-7)
    nwd_mat = torch.exp(-w2 / C)
    best_nwd = nwd_mat.max(dim=0).values
    return (1.0 - best_nwd).mean()


# ── Detection head ────────────────────────────────────────────
class DetectionHead(nn.Module):
    def __init__(self, num_classes, num_queries=100):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(8),
        )
        feat_dim = 256 * 8 * 8
        self.box_head = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.ReLU(True),
            nn.Linear(512, num_queries * 4))
        self.cls_head = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.ReLU(True),
            nn.Linear(512, num_queries * num_classes))
        self.num_queries  = num_queries
        self.num_classes  = num_classes

    def forward(self, x):
        B    = x.shape[0]
        feat = self.backbone(x).view(B, -1)
        boxes   = torch.sigmoid(
            self.box_head(feat).view(B, self.num_queries, 4))
        logits  = self.cls_head(feat).view(
            B, self.num_queries, self.num_classes)
        return boxes, logits


# ── ESRGAN builder (half=False for AMP) ──────────────────────
def build_esrgan(scale, device):
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from basicsr.utils.download_util import load_file_from_url
    url = (
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"
        if scale == 2 else
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
    )
    path = load_file_from_url(url, "/root/.cache/realesrgan",
                               progress=True)
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                    num_block=23, num_grow_ch=32, scale=scale)
    return RealESRGANer(scale=scale, model_path=path,
                        model=model, tile=0, tile_pad=10,
                        pre_pad=0, half=False, device=device)


# ── Trainer ───────────────────────────────────────────────────
class JointNWDTrainer:

    def __init__(self, sr_scale, num_classes, device,
                 lr_esrgan, lr_detect, lambda_sr, nwd_c):
        self.device    = device
        self.best_loss = 1e9
        self.lambda_sr = lambda_sr
        self.nwd_c     = nwd_c
        self.num_cls   = num_classes
        self.history   = {"L_nwd":[],"L_sr":[],
                          "L_cls":[],"L_total":[]}

        print(f"  Loading ESRGAN {sr_scale}×...")
        self.esrgan = build_esrgan(sr_scale, device).model.to(device)

        print(f"  Building DetectionHead ({num_classes} classes)...")
        self.det_head = DetectionHead(
            num_classes, NUM_QUERIES).to(device)

        self.l1_loss  = nn.L1Loss()
        self.cls_loss = nn.CrossEntropyLoss(ignore_index=-1)

        self.opt_esrgan = torch.optim.AdamW(
            self.esrgan.parameters(), lr=lr_esrgan)
        self.opt_det    = torch.optim.AdamW(
            self.det_head.parameters(), lr=lr_detect)

    def training_step(self, batch):
        lr    = batch["lr"].to(self.device)
        hr    = batch["hr"].to(self.device)
        boxes = batch["boxes"].to(self.device)  # (B, MAX_BOXES, 5)

        # ESRGAN: LR → SR
        sr = self.esrgan(lr)
        if sr.shape[-2:] != hr.shape[-2:]:
            sr = F.interpolate(sr, size=hr.shape[-2:],
                               mode="bilinear", align_corners=False)

        # SR quality loss
        L_sr = self.l1_loss(sr, hr)

        # Detection head
        pred_boxes, pred_logits = self.det_head(sr)

        # NWD + classification per image
        L_nwd = torch.tensor(0.0, device=self.device)
        L_cls = torch.tensor(0.0, device=self.device)
        B = lr.shape[0]
        valid_count = 0

        for b in range(B):
            gt   = boxes[b]               # (MAX_BOXES, 5)
            mask = gt[:, 0] >= 0          # valid rows
            if not mask.any():
                continue
            gt_valid  = gt[mask]          # (M, 5)
            gt_cls    = gt_valid[:, 0].long()
            gt_coords = gt_valid[:, 1:]   # (M, 4) cx cy w h

            gt_cls = torch.clamp(gt_cls, 0, self.num_cls - 1)

            pb = pred_boxes[b]    # (Q, 4)
            pl = pred_logits[b]   # (Q, C)

            # True NWD
            L_nwd = L_nwd + nwd_loss(pb, gt_coords, self.nwd_c)

            # Cls: each query → nearest GT by NWD
            M = gt_coords.shape[0]
            mu_p  = pb[:, :2]; sig_p = pb[:, 2:] / 2
            mu_g  = gt_coords[:, :2]; sig_g = gt_coords[:, 2:] / 2
            cd = ((mu_p.unsqueeze(1)-mu_g.unsqueeze(0))**2).sum(-1)
            sd = ((sig_p.unsqueeze(1)-sig_g.unsqueeze(0))**2).sum(-1)
            matched = torch.exp(
                -torch.sqrt(cd+sd+1e-7)/self.nwd_c).argmax(dim=1)
            L_cls = L_cls + self.cls_loss(pl, gt_cls[matched])
            valid_count += 1

        if valid_count > 0:
            L_nwd = L_nwd / valid_count
            L_cls = L_cls / valid_count

        L_total = 2.0*L_nwd + 1.0*L_cls + self.lambda_sr*L_sr

        return {"L_total": L_total,
                "L_nwd":  L_nwd.detach(),
                "L_cls":  L_cls.detach(),
                "L_sr":   L_sr.detach()}

    def save(self, path, epoch):
        torch.save({"epoch"   : epoch,
                    "esrgan"  : self.esrgan.state_dict(),
                    "det_head": self.det_head.state_dict(),
                    "best_loss": self.best_loss,
                    "history" : self.history}, path)

print("✅ Cell 11 ready — all classes defined")
print(f"   Experiments: {[e['name'] for e in EXPERIMENTS]}")

✅ Cell 11 ready — all classes defined
   Experiments: ['VisDrone_2x', 'VisDrone_4x', 'AITOD_2x', 'AITOD_4x']


In [ ]:
# ============================================================
# CELL 12 — Run all 4 joint training experiments
# ORDER: VisDrone 2x → VisDrone 4x → AITOD 2x → AITOD 4x
# ============================================================

import gc, torch
from torch.utils.data import DataLoader
from pathlib import Path

# Safe torch.load patch
if not hasattr(torch, "_load_patched"):
    _orig = torch.load
    def _patched(*a, **kw):
        kw["weights_only"] = False
        return _orig(*a, **kw)
    torch.load = _patched
    torch._load_patched = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALL_TRAINERS = {}

for exp in EXPERIMENTS:

    print("\n" + "="*65)
    print(f"  EXPERIMENT : {exp['name']}")
    print(f"  SR Scale   : {exp['sr_scale']}×")
    print(f"  Classes    : {exp['num_classes']}")
    print(f"  Epochs     : {exp['epochs']}")
    print("="*65)

    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # ── Dataset ───────────────────────────────────────────────
    train_ds = JointTileDataset(
        img_dir   = exp["tile_dir"] / "train" / "images",
        lbl_dir   = exp["tile_dir"] / "train" / "labels",
        sr_scale  = exp["sr_scale"],
        max_boxes = MAX_BOXES,
    )

    if len(train_ds) == 0:
        print(f"  ❌ No tiles found in "
              f"{exp['tile_dir']}/train/images — skipping")
        ALL_TRAINERS[exp["name"]] = None
        continue

    train_loader = DataLoader(
        train_ds,
        batch_size  = BATCH_SIZE,
        shuffle     = True,
        num_workers = NUM_WORKERS,
        pin_memory  = True,
    )
    print(f"  Batches/epoch : {len(train_loader)}")

    # ── Trainer ───────────────────────────────────────────────
    trainer = JointNWDTrainer(
        sr_scale    = exp["sr_scale"],
        num_classes = exp["num_classes"],
        device      = DEVICE,
        lr_esrgan   = LR_ESRGAN,
        lr_detect   = LR_DETECT,
        lambda_sr   = LAMBDA_SR,
        nwd_c       = NWD_C,
    )

    scaler   = torch.amp.GradScaler("cuda")
    ckpt_dir = Path("/kaggle/working/runs") / exp["name"]
    ckpt_dir.mkdir(exist_ok=True)

    # ── Epoch loop ────────────────────────────────────────────
    for epoch in range(exp["epochs"]):
        trainer.esrgan.train()
        trainer.det_head.train()

        running = {"L_total":0.0,"L_nwd":0.0,
                   "L_cls":0.0,"L_sr":0.0}

        trainer.opt_esrgan.zero_grad()
        trainer.opt_det.zero_grad()

        loop = tqdm(train_loader,
                    desc=f"{exp['name']} "
                         f"Ep{epoch+1}/{exp['epochs']}")

        for step, batch in enumerate(loop):
            try:
                with torch.amp.autocast("cuda"):
                    m    = trainer.training_step(batch)
                    loss = m["L_total"] / GRAD_ACCUM

                scaler.scale(loss).backward()

                if (step + 1) % GRAD_ACCUM == 0:
                    scaler.step(trainer.opt_esrgan)
                    scaler.step(trainer.opt_det)
                    scaler.update()
                    trainer.opt_esrgan.zero_grad()
                    trainer.opt_det.zero_grad()

                for k in running:
                    v = m[k]
                    running[k] += (v.item()
                                   if isinstance(v, torch.Tensor)
                                   else float(v))

                loop.set_postfix({
                    "tot": f"{running['L_total']/(step+1):.4f}",
                    "nwd": f"{running['L_nwd']/(step+1):.4f}",
                    "sr" : f"{running['L_sr']/(step+1):.4f}",
                })

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print("\n  ⚠️  OOM — skipping batch")
                    torch.cuda.empty_cache()
                    gc.collect()
                    continue
                raise

        # Epoch summary + history
        N  = max(1, len(train_loader))
        ep = {k: v/N for k, v in running.items()}
        for k, v in ep.items():
            trainer.history[k].append(v)

        print(f"\n  Ep{epoch+1}  "
              f"tot={ep['L_total']:.4f}  "
              f"nwd={ep['L_nwd']:.4f}  "
              f"cls={ep['L_cls']:.4f}  "
              f"sr={ep['L_sr']:.4f}")

        # Save best
        if ep["L_total"] < trainer.best_loss:
            trainer.best_loss = ep["L_total"]
            trainer.save(ckpt_dir/"best.pt", epoch+1)
            print("  ✅ Best saved")

    ALL_TRAINERS[exp["name"]] = trainer
    print(f"\n✅ {exp['name']} done | "
          f"best_loss={trainer.best_loss:.4f}")

print("\n🎉 ALL EXPERIMENTS COMPLETE")


  EXPERIMENT : VisDrone_2x
  SR Scale   : 2×
  Classes    : 6
  Epochs     : 10
   [train/images]: 3459 object tiles  (LR 320×320)
  Batches/epoch : 3459
  Loading ESRGAN 2×...
  Building DetectionHead (6 classes)...


VisDrone_2x Ep1/10:   0%|          | 0/3459 [00:00<?, ?it/s]


  Ep1  tot=1.3132  nwd=0.0066  cls=1.2900  sr=0.0992
  ✅ Best saved


VisDrone_2x Ep2/10:   0%|          | 0/3459 [00:00<?, ?it/s]


  Ep2  tot=1.2127  nwd=0.0055  cls=1.1903  sr=0.1139
  ✅ Best saved


...

AITOD_2x Ep8/10:   0%|          | 0/5830 [00:00<?, ?it/s]


  Ep8  tot=0.1750  nwd=0.0042  cls=0.1592  sr=0.0728
  ✅ Best saved


AITOD_2x Ep9/10:   0%|          | 0/5830 [00:00<?, ?it/s]


  Ep9  tot=0.1453  nwd=0.0042  cls=0.1307  sr=0.0635
  ✅ Best saved


AITOD_2x Ep10/10:   0%|          | 0/5830 [00:00<?, ?it/s]

In [5]:
# ============================================================
# CELL — Check saved checkpoints
# ============================================================

import torch
from pathlib import Path

RUNS_DIR = Path("/kaggle/working/runs")

print("="*60)
print("  CHECKPOINT STATUS")
print("="*60)

for exp_name in ["VisDrone_2x","VisDrone_4x",
                 "AITOD_2x","AITOD_4x"]:

    ckpt_dir  = RUNS_DIR / exp_name
    best_pt   = ckpt_dir / "best.pt"

    if not ckpt_dir.exists():
        print(f"\n  ❌ {exp_name:<15} — folder missing")
        continue

    if not best_pt.exists():
        print(f"\n  ❌ {exp_name:<15} — no best.pt found")
        continue

    # Load and inspect checkpoint
    ckpt = torch.load(str(best_pt),
                      map_location="cpu",
                      weights_only=False)
    epoch     = ckpt.get("epoch",     "?")
    best_loss = ckpt.get("best_loss", "?")
    history   = ckpt.get("history",   {})
    n_epochs  = len(history.get("L_total", []))

    print(f"\n  ✅ {exp_name:<15}")
    print(f"     Saved at epoch : {epoch}")
    print(f"     Best loss      : {best_loss:.4f}"
          if isinstance(best_loss, float) else
          f"     Best loss      : {best_loss}")
    print(f"     History length : {n_epochs} epochs")

    keys = list(ckpt.keys())
    print(f"     Checkpoint keys: {keys}")

print("\n" + "="*60)

  CHECKPOINT STATUS

  ❌ VisDrone_2x     — folder missing

  ❌ VisDrone_4x     — folder missing

  ❌ AITOD_2x        — folder missing

  ❌ AITOD_4x        — folder missing



In [6]:
# ============================================================
# VISUALIZATION — Run any time during or after training
# Reads ALL_TRAINERS that have completed so far
# ============================================================
  
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# ── Colors per experiment ─────────────────────────────────── ──
COLORS = { 
    "VisDrone_2x": "#3498DB",
    "VisDrone_4x": "#2ECC71",
    "AITOD_2x"   : "#E74C3C",
    "AITOD_4x"   : "#F39C12",
}

# ── Collect history from completed trainers ───────────────────
completed = {k: v for k, v in ALL_TRAINERS.items()
             if v is not None and v.history["L_total"]}

if not completed:
    print("No completed experiments yet — run after at least 1 epoch")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.patch.set_facecolor("#0F1117")
    axes = axes.flatten()

    LOSS_KEYS  = ["L_total", "L_nwd", "L_cls", "L_sr"]
    LOSS_TITLES= ["Total Combined Loss",
                  "NWD Box Loss",
                  "Classification Loss",
                  "SR Quality Loss"]

    for ax, lkey, ltitle in zip(axes, LOSS_KEYS, LOSS_TITLES):
        ax.set_facecolor("#1A1D27")
        ax.set_title(ltitle, color="white",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Epoch", color="white", fontsize=9)
        ax.set_ylabel("Loss",  color="white", fontsize=9)
        ax.tick_params(colors="white")
        ax.spines[["top","right","left","bottom"]].set_color("#333644")
        ax.yaxis.grid(True, color="#2A2D3A",
                      linewidth=0.8, linestyle="--")
        ax.set_axisbelow(True)
  
        for exp_name, trainer in completed.items():
            vals = trainer.history.get(lkey, [])
            if not vals: continue
            col  = COLORS.get(exp_name, "gray")
            eps  = range(1, len(vals)+1)
            ax.plot(eps, vals, color=col, linewidth=2.5,
                    label=exp_name, marker="o", markersize=5,
                    markerfacecolor="white", markeredgecolor=col)
            ax.fill_between(eps, vals, alpha=0.08, color=col)
            # Annotate final value
            ax.annotate(f"{vals[-1]:.4f}",
                        (len(vals), vals[-1]),
                        textcoords="offset points",
                        xytext=(6, 0), color=col,
                        fontsize=8, fontweight="bold")

        ax.legend(fontsize=8, facecolor="#1A1D27",
                  labelcolor="white", edgecolor="#333644")

    plt.suptitle(
        "Joint ESRGAN + RT-DETR Training — Loss Curves\n"
        "VisDrone Tiny + AI-TOD | NWD Loss",
        color="white", fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("/kaggle/working/loss_curves.png",
                dpi=150, bbox_inches="tight",
                facecolor="#0F1117")
    plt.show()
    print("✅ Saved → /kaggle/working/loss_curves.png")

NameError: name 'ALL_TRAINERS' is not defined

In [ ]:
# ============================================================
# VISUALIZATION 2 — Per-experiment loss breakdown bar chart
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor("#0F1117")

exp_names  = list(completed.keys())       
short_names= [n.replace("VisDrone","VD").replace("AITOD","AT")
              for n in exp_names]
colors     = [COLORS.get(n,"gray") for n in exp_names]

# Final epoch losses for each component
final_nwd = [completed[n].history["L_nwd"][-1]
             for n in exp_names]
final_cls = [completed[n].history["L_cls"][-1]
             for n in exp_names]
final_sr  = [completed[n].history["L_sr"][-1]
             for n in exp_names]
final_tot = [completed[n].history["L_total"][-1]
             for n in exp_names]

x = np.arange(len(exp_names))
w = 0.22

# ── LEFT — stacked loss components ───────────────────────────
ax = axes[0]
ax.set_facecolor("#1A1D27")
b1 = ax.bar(x, final_nwd, w*3, label="NWD",
            color="#4C9BE8", edgecolor="#0F1117")
b2 = ax.bar(x, final_cls, w*3, bottom=final_nwd,
            label="Classification",
            color="#E8734C", edgecolor="#0F1117")
b3 = ax.bar(x, [s*0.1 for s in final_sr], w*3,
            bottom=[n+c for n,c in zip(final_nwd,final_cls)],
            label="SR (×0.1 weight)",
            color="#4CE87A", edgecolor="#0F1117")

ax.set_xticks(x)
ax.set_xticklabels(short_names, color="white", fontsize=10)
ax.set_ylabel("Loss (final epoch)", color="white", fontsize=10)
ax.set_title("Loss Component Breakdown\n(Final Epoch)",
             color="white", fontsize=12, fontweight="bold")
ax.tick_params(colors="white")
ax.spines[["top","right","left","bottom"]].set_color("#333644")
ax.yaxis.grid(True, color="#2A2D3A", linewidth=0.8,
              linestyle="--", zorder=0)
ax.set_axisbelow(True)
ax.legend(fontsize=9, facecolor="#1A1D27",
          labelcolor="white", edgecolor="#333644")

# ── RIGHT — total loss convergence across epochs ──────────────
ax2 = axes[1]
ax2.set_facecolor("#1A1D27")
for exp_name, trainer in completed.items():
    vals = trainer.history["L_total"]
    eps  = range(1, len(vals)+1)
    col  = COLORS.get(exp_name, "gray")
    ax2.plot(eps, vals, color=col, linewidth=2.5,
             label=exp_name.replace("VisDrone","VD")
                            .replace("AITOD","AT"),
             marker="o", markersize=6,
             markerfacecolor="white")
    # Shade convergence zone
    if len(vals) >= 2:
        ax2.annotate(f"  {vals[-1]:.4f}",
                     (len(vals), vals[-1]),
                     color=col, fontsize=9,
                     fontweight="bold")

ax2.set_xlabel("Epoch", color="white", fontsize=10)
ax2.set_ylabel("Total Loss",  color="white", fontsize=10)
ax2.set_title("Total Loss Convergence",
              color="white", fontsize=12, fontweight="bold")
ax2.tick_params(colors="white")
ax2.spines[["top","right","left","bottom"]].set_color("#333644")
ax2.yaxis.grid(True, color="#2A2D3A", linewidth=0.8,
               linestyle="--")
ax2.set_axisbelow(True)
ax2.legend(fontsize=9, facecolor="#1A1D27",
           labelcolor="white", edgecolor="#333644")

plt.suptitle(
    "Training Summary — Joint ESRGAN + RT-DETR + NWD",
    color="white", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig("/kaggle/working/training_summary.png",
            dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
print("✅ Saved → /kaggle/working/training_summary.png")

In [ ]:
# ============================================================
# VISUALIZATION 3 — Sample SR quality: LR vs SR output
# Run after at least VisDrone_2x finishes
# ============================================================

import cv2, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

VD_TILES = Path("/kaggle/working/vd_tiles")
DEVICE   = torch.device("cuda" if torch.cuda.is_available()
                         else "cpu")

# Get 3 sample val tiles
val_imgs = sorted((VD_TILES/"val"/"images").glob("*.jpg"))[:3]
if not val_imgs:
    print("No val tiles found yet")
else:
    # Use the best trained ESRGAN from VisDrone_2x
    trainer_vd2 = ALL_TRAINERS.get("VisDrone_2x")

    fig, axes = plt.subplots(3, 3, figsize=(15, 13))
    fig.patch.set_facecolor("#0F1117")

    for row, img_path in enumerate(val_imgs):
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        H, W    = img_rgb.shape[:2]

        # Col 0: Original HR tile
        axes[row][0].imshow(img_rgb)
        axes[row][0].set_title("Original 640×640",
                               color="white", fontsize=9)

        # Col 1: Downsampled LR (what ESRGAN sees)
        lr = cv2.resize(img_rgb, (W//2, H//2),
                        interpolation=cv2.INTER_AREA)
        lr_up = cv2.resize(lr, (W, H),
                           interpolation=cv2.INTER_NEAREST)
        axes[row][1].imshow(lr_up)
        axes[row][1].set_title("LR Input (320×320 → shown at 640)",
                               color="#E8734C", fontsize=9)

        # Col 2: ESRGAN SR output
        if trainer_vd2 is not None:
            try:
                lr_t = torch.from_numpy(
                    lr.astype(np.float32)/255.0
                ).permute(2,0,1).unsqueeze(0).to(DEVICE)

                trainer_vd2.esrgan.eval()
                with torch.no_grad():
                    sr_t = trainer_vd2.esrgan(lr_t)
                trainer_vd2.esrgan.train()

                sr_np = sr_t.squeeze(0).permute(1,2,0)\
                            .clamp(0,1).cpu().numpy()
                sr_np = (sr_np * 255).astype(np.uint8)
                if sr_np.shape[:2] != (H, W):
                    sr_np = cv2.resize(sr_np, (W, H))
                axes[row][2].imshow(sr_np)
                axes[row][2].set_title(
                    "ESRGAN 2× SR Output",
                    color="#4CE87A", fontsize=9,
                    fontweight="bold")
            except Exception as e:
                axes[row][2].text(0.5, 0.5, f"Error:\n{e}",
                                  color="white", ha="center",
                                  va="center",
                                  transform=axes[row][2].transAxes,
                                  fontsize=7)
        else:
            axes[row][2].text(0.5, 0.5,
                              "VisDrone_2x\nnot yet complete",
                              color="white", ha="center",
                              va="center",
                              transform=axes[row][2].transAxes)

        for ax in axes[row]:
            ax.axis("off")
            ax.set_facecolor("#1A1D27")

    plt.suptitle(
        "SR Quality: Original → LR Input → ESRGAN Output\n"
        "VisDrone Tiny Val Tiles",
        color="white", fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("/kaggle/working/sr_quality.png",
                dpi=150, bbox_inches="tight",
                facecolor="#0F1117")
    plt.show()
    print("✅ Saved → /kaggle/working/sr_quality.png")

In [ ]:
# ============================================================
# VISUALIZATION 4 — Text summary table (no mAP yet)
# Shows training progress for all completed experiments
# ============================================================

print("\n" + "="*70)
print("  TRAINING PROGRESS SUMMARY")
print("="*70)
print(f"  {'Experiment':<18} {'Epochs':>7} {'Init Loss':>10} "
      f"{'Final Loss':>11} {'Best Loss':>10} {'Drop %':>8}")
print("  " + "-"*67)

for exp_name, trainer in ALL_TRAINERS.items():
    if trainer is None or not trainer.history["L_total"]:
        print(f"  {exp_name:<18} {'not started':>35}")
        continue
    hist   = trainer.history["L_total"]
    init   = hist[0]
    final  = hist[-1]
    best   = trainer.best_loss
    drop   = (init - final) / init * 100
    epochs = len(hist)
    print(f"  {exp_name:<18} {epochs:>7} {init:>10.4f} "
          f"{final:>11.4f} {best:>10.4f} {drop:>7.1f}%")

print("="*70)
print("\n  NWD loss trend (lower = better box alignment):")
for exp_name, trainer in ALL_TRAINERS.items():
    if trainer is None or not trainer.history["L_nwd"]:
        continue
    nwd = trainer.history["L_nwd"]
    trend = " → ".join([f"{v:.4f}" for v in nwd[::max(1,len(nwd)//4)]])
    print(f"  {exp_name:<18}: {trend}")
print("="*70)